In [58]:
import os, json, tempfile
import pandas as pd
import ydb
import ydb.iam
from dotenv import load_dotenv
import yaml

# 1) Подхватываем .env из текущей папки (и можно указать путь)
load_dotenv(".env", override=False)

# 2) Читаем db.yaml
with open("../config/db.yaml", "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

# Выбирай логическое имя таблицы из yaml: tasks, task_versions, sync_state, ...
TABLE_ALIAS = "tasks"
TABLE = cfg["tables"][TABLE_ALIAS]  # например "dtm_tasks"

# 3) Берем тестовые переменные
ENDPOINT = os.environ["YDB_ENDPOINT_TEST"]
DATABASE = os.environ["YDB_DATABASE_TEST"]
SA = os.environ["YC_SA_JSON_CREDENTIALS"]  # JSON-строка или путь к файлу
i = ENDPOINT.find("/?")
if i > 0:
    ENDPOINT = ENDPOINT[:i]
def creds_from_sa(value: str):
    v = value.strip()
    if v.startswith("{"):  # JSON-строка
        fd, path = tempfile.mkstemp(prefix="yc-sa-", suffix=".json")
        os.close(fd)
        with open(path, "w", encoding="utf-8") as f:
            json.dump(json.loads(v), f)
        return ydb.iam.ServiceAccountCredentials.from_file(path)
    return ydb.iam.ServiceAccountCredentials.from_file(v)

driver_config = ydb.DriverConfig(
    ENDPOINT,
    DATABASE,
    credentials=creds_from_sa(SA),
    root_certificates=ydb.load_ydb_root_certificate(),
)

sql = f"SELECT * FROM `{TABLE}` LIMIT 1000"

with ydb.Driver(driver_config) as driver:
    driver.wait(timeout=20)
    with ydb.QuerySessionPool(driver,size=20) as pool:
        rs = pool.execute_with_retries(sql)

df = pd.DataFrame.from_records(rs[0].rows if rs else [])
print(TABLE, df.shape)
# print(df.head())

dtm_tasks (448, 19)


In [59]:
print(df.status.value_counts())
df.sort_values('end_date', inplace=True, ascending=False)
df.head()

status
done    443
work      5
Name: count, dtype: int64


,brand,customer,end_date,format_,group_id,history,links_json,next_due_date,owner_id,raw_payload,raw_timing,start_date,status,tags_json,task_hash,task_id,task_revision,title,updated_at_utc
400,Вологодский пломбир,Носенко Василий,2026-04-12,граф. ролик,Сокровища Императора,"20.02 - отдали раскадровку, вернулись коммента...",{},2026-02-20,Упадышев Денис,"{""task_id"": ""61baabb2-ef03-4593-a6c0-82bf6bfce...",20.02 - раскадровка\r\n25.02 - ответ\r\n13.03 ...,2026-02-20,work,[],df691b570ab12c31be8417f032a70cb2e5e491cfa1c363...,61baabb2-ef03-4593-a6c0-82bf6bfcedb8,1,Пломбир [СОКР ИМП] граф. ролик,2026-03-05 13:41:24.232606
0,Cool Cola,Носенко Василий,2026-04-07,граф. ролик,Сокровища Императора,13.02 - отправили сториборд,{},2026-02-13,студия Громко,"{""task_id"": ""00017f0b-b9b2-44b6-8c7f-9efd5df5b...",13.02 - визуальный сценарий (сториборд в скетч...,2026-02-13,work,[],ca5369f212bc5b8d836cd76e0294a70a9e59fca9c97c7c...,00017f0b-b9b2-44b6-8c7f-9efd5df5bf7a,1,Cool Cola [СОКР ИМП] граф. ролик,2026-03-05 13:41:24.232606
105,Joonies,Носенко Василий,2026-04-03,граф. ролик,Сокровища Императора,16.02 - отдали раскадровку \n26.02 - прислали ...,{},2026-02-20,Левашова Лера,"{""task_id"": ""1aab054a-b39e-414c-b377-11b769fb1...",20.02 - раскадровка\r\n27.02 - ответ\r\n12.03 ...,2026-02-20,done,[],39820a715589f23684e716696dcdf48172ac37822ec8a6...,1aab054a-b39e-414c-b377-11b769fb11d0,1,Joonies [СОКР ИМП] граф. ролик,2026-03-05 13:41:24.232606
348,Момат рино,Русина Александра,2026-03-17,граф.ролик,Выжить в Стамбуле,04.03 - отдали арскадровку,{},2026-03-04,Муратов Эдуард,"{""task_id"": ""54d23f38-c554-4b32-b5cd-7cf5e5a0d...",04.03 - раскадровка\r\n05.03 - ответ\r\n11.03 ...,2026-03-04,work,[],bee8724c8e85ebbbb4d8946eb237648ac784bd1e6cdd49...,54d23f38-c554-4b32-b5cd-7cf5e5a0de28,1,Момат рино [СТАМБУЛ] граф.ролик,2026-03-05 13:41:24.232606
38,Момат рино,Русина Александра,2026-03-10,дин лого,Выжить в Стамбуле,26.02 - отдали раскадровку \n04.03 - вернулись...,{},2026-02-25,Статкевич Андрей,"{""task_id"": ""0a4a86d3-457c-4eb6-8028-720b3593b...",25.02 - раскадровка\r\n26.02 - ответ клиента\r...,2026-02-25,work,[],2a2c40b1c5f538e1a4f6635d455a03cfb238a11bef5c8a...,0a4a86d3-457c-4eb6-8028-720b3593b3b9,1,Момат рино [СТАМБУЛ] дин лого,2026-03-05 13:41:24.232606


In [ ]:

class RawRepository:
    def __init__(self, source, cache, normalizer, hasher):
        self.source = source
        self.casche = cache
        self.normalizer = normalizer
        self.hasher = hasher       
        self.hashe = cache.get_hashe()

    def update(self):
        data = self.source.get()
        old_hahe = self.hashe
        new_hashe = self.hasher(data)
        if old_hahe != new_hashe:            
            norm_data = self.normalizer(data)
            self.casche.set(norm_data, new_hashe)
        
        return {"hashe": new_hashe,
                "changed": old_hahe != new_hashe,
        }
            
    def get(self):
        return self.casche.get()


class PrepRepository:
    def __init__(self, source, cache, preprocessor, hasher):
        self.source = source
        self.casche = cache
        self.preprocessor = preprocessor
        self.hasher = hasher 
        self.hashe = cache.get_hashe()


    
    def update(self):
        raw_info = self.source.update()
        if raw_info["changed"]:
            raw_data = self.source.get()
            old_data = self.casche.get()        
            new_ids, deleted_ids, changed_ids, unchanged_ids = self.diff.diff(old_data, raw_data)            
            data = self.preprocessor(old_data, raw_data, new_ids, deleted_ids, changed_ids, unchanged_ids)
            self.casche.set(data, raw_info["hashe"])

        return {
            "hashe": raw_info["hashe"],
            "changed": raw_info["changed"],
        }
    
    def get(self):
        return self.casche.get()

class Processor:
    def __init__(self, cfg, diff):
        self.cfg = cfg
        self.diff = diff

    def process(self, old_data, raw_data):
        new_ids, deleted_ids, changed_ids, unchanged_ids = self.diff(old_data, raw_data)

class Diff:
    def __init__(self, cfg, hasher):
        self.cfg = cfg
        self.hasher = hasher

    def diff(self, old_data, raw_data):
        # TODO: логика сравнения данных и выделения новых, удаленных, измененных и неизменных записей
        return new_ids, deleted_ids, changed_ids, unchanged_ids


class Updater:
    def __init__(self, source, cfg):
        self.source = source
        self.rule = self.make_from_cfg(self, cfg)
    
    def work(self, request):
        if self.rule():
            return self.source.update()
    
    def make_from_cfg(self, cfg): 
        rule = None
        # TODO: логика создания правила на основе cfg
        return rule


class DataMart:
    def __init__(self, source, filter):
        self.source = source
        self.filter = filter

    def get(self,query):
        data = self.source.get()
        ids = self.filter(data, query)
        return data[ids]

   
class Worker:
    def __init__(self, source, cfg):        
        self.cfg = cfg
        self.source = source
        self.formatter, self.service = self.make_from_cfg(cfg)
    
    def make_from_cfg(self, cfg):         
        source = None
        formatter = None
        service = None
        return source, formatter, service

    def make_query(self):        
        return None

    def prepare(self, request):
        query = self.make_query(request)
        data = self.source.get(query)        
        return self.formatter(data)
    
    def work(self, request):
        data = self.prepare(request)
        return self.service.render(data)
    
    def __call__(self, request):
        return self.work(request)


class Render(Worker):
    def make_query(self, request):        
        cfg = self.cfg
        query = None
        # TODO: 
        # логика формирования запроса на основе cfg
        return query
    
    def make_from_cfg(self, cfg):         
        source = None
        formatter = None
        service = None
        # TODO: логика создания source, formatter, service на основе cfg
        return source, formatter, service


class Notifier(Worker):    
    def make_query(self, request):
        cfg = self.cfg
        query = None
        # TODO: 
        # логика формирования запроса на основе cfg
        return query
    
    def make_from_cfg(self, cfg):         
        source = None
        formatter = None
        service = None
        # TODO: логика создания source, formatter, service на основе cfg
        return source, formatter, service


class Api(Worker):    
    def make_query(self, request):
        cfg = self.cfg
        query = None
        # TODO: 
        # логика формирования запроса на основе cfg
        return query
    
    def make_from_cfg(self, cfg):         
        source = None
        formatter = None
        service = None
        # TODO: логика создания source, formatter, service на основе cfg
        return source, formatter, service

            
def main(request):
    mode = parse_mode(request)

    workers = {
        "render": Render(source, cfg.render),
        "notify": Notifier(source, cfg.notify),
        "api": Api(source, cfg.api),
        "update": Updater(source, cfg.update),
    }
    worker = workers[mode]
    
    return worker.work(request)


    


In [1]:
import os
from dotenv import load_dotenv

load_dotenv(".env")

required = [
    "GITHUB_TOKEN", "GITHUB_OWNER", "GITHUB_REPO",
    "JIRA_BASE_URL", "JIRA_EMAIL", "JIRA_API_TOKEN", "JIRA_PROJECT_KEY"
]
missing = [k for k in required if not os.getenv(k)]
if missing:
    raise RuntimeError(f"Missing in .env: {missing}")

print("OK: env loaded")
print("GitHub:", f"{os.getenv('GITHUB_OWNER')}/{os.getenv('GITHUB_REPO')}")
print("Jira:", os.getenv("JIRA_BASE_URL"), "Project:", os.getenv("JIRA_PROJECT_KEY"))

OK: env loaded
GitHub: yrsolo/DTM
Jira: https://yrsolo.atlassian.net/ Project: DTM


In [ ]:



from warnings import filters


if mode==modes.timer:
    repository.check_for_updates()

elif mode==modes.render:
    tasks = repository.get_tasks('active')
    result = render_engine.render(tasks)

elif mode==modes.reminder:
    tasks = repository.get_tasks('active', for_days=[now,tomorrow])
    send_reminders(tasks)

elif mode==modes.api:
    filters = parse_api_request(request)
    tasks = get_tasks(filters)
    return tasks


class Repository:
    ...
def get_tasks(self, status, for_days=None):
    last_time_updated = self.get_last_time_updated()
    if last_time_updated > min_refresh_time:
        self.refresh()
    tasks = self.fetch_tasks(status, for_days)

    return tasks

def refresh(self):
    if last_time_updated > max_refresh_time:
        self.fetch_from_source()
    else:
        


        